# Stage 3b — How to feed commits to CodeBERT

*(Printed output below still calls this Stage 3.5, its earlier name.)*

Four questions, answered in order. Each measurement feeds the next, and the decision comes last.

1. **Is 512 tokens enough?**
2. **How did Levin use ChangeDistiller?**
3. **How do we mimic that with CodeBERT?**
4. **Why not just keep the raw code changes and drop everything else?**

*Its outputs were renamed for publication: the printed output below says `stage35_decision.json` and `stage35_token_counts.csv`, which the code now writes as `stage3b_decision.json` and `stage3b_token_counts.csv`.*

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 0. Load

Stage 3a's output, with the two rows it flagged for exclusion removed: one commit the API truncated
at 300 files, one touching no Java file.

In [1]:
import json, re, collections
from pathlib import Path
import pandas as pd
import numpy as np

SRC = require("levin_with_diffs.jsonl")      # Stage 3a output, or the published copy

R = pd.DataFrame([json.loads(l) for l in open(SRC, encoding="utf-8") if l.strip()])
R = R[R["ok"] == True].reset_index(drop=True)

R["java_files"] = R["files"].map(
    lambda fs: sum(1 for f in (fs or []) if str(f.get("filename", "")).endswith(".java")))
drop = R[(R["truncated"] == True) | (R["java_files"] == 0)]
R = R[~R.index.isin(drop.index)].reset_index(drop=True)
print(f"{len(R):,} commits under measurement ({len(drop)} excluded)")

MODEL, LIMIT = "microsoft/codebert-base", 512
try:
    from transformers import AutoTokenizer
    TOK = AutoTokenizer.from_pretrained(MODEL)
    def ntok(t): return len(TOK.encode(t, add_special_tokens=True, truncation=False))
    print(f"tokeniser: {MODEL}, limit {LIMIT}")
except Exception as e:
    TOK = None
    def ntok(t): return len(t) // 4 + 2
    print(f"transformers unavailable ({type(e).__name__}); using ~4 chars/token. "
          f"Relative comparisons hold, absolute counts do not.")

HUNK = re.compile(r"^@@ .*? @@")

def message_of(rec):
    """Git's message. Stage 3a found the dataset's `comment` substitutes '-' for newlines and
    '#', which would match nothing seen at application time."""
    return (rec.get("git_message") or rec.get("dataset_comment") or "").strip()

1,150 commits under measurement (1 excluded)


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

tokeniser: microsoft/codebert-base, limit 512


## 1. Is 512 tokens enough?

The whole commit — message plus every changed file — as one input.

In [2]:
def rep_raw(rec):
    parts = [message_of(rec)]
    for f in rec.get("files") or []:
        parts.append(f"--- {f.get('filename')}")
        if f.get("patch"):
            parts.append(f["patch"])
    return "\n".join(parts)


raw_tokens = R.apply(lambda r: ntok(rep_raw(r)), axis=1)
print("TOKENS PER COMMIT, raw unified diff\n")
print(raw_tokens.describe(percentiles=[.5, .75, .9, .95, .99]).round(0).to_string())
print(f"\n   within {LIMIT} tokens : {(raw_tokens <= LIMIT).mean()*100:.1f}%")
print(f"   over                : {(raw_tokens > LIMIT).mean()*100:.1f}%")
print(f"\nAnswer: no. Most commits do not fit, and the largest is "
      f"{int(raw_tokens.max()):,} tokens -- {raw_tokens.max()/LIMIT:.0f} times the limit.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (12012 > 512). Running this sequence through the model will result in indexing errors


TOKENS PER COMMIT, raw unified diff

count     1150.0
mean      4868.0
std      10009.0
min        117.0
50%       1670.0
75%       4303.0
90%      11491.0
95%      18734.0
99%      62052.0
max      90669.0

   within 512 tokens : 20.2%
   over                : 79.8%

Answer: no. Most commits do not fit, and the largest is 90,669 tokens -- 177 times the limit.


## 2. How did Levin use ChangeDistiller?

From §5.2 of the paper:

> *we repeatedly applied ChangeDistiller on every two consecutive revisions of every Java file in
> every repository we had selected*

And from §1, what it produces:

> *Fluri's taxonomy of source code changes for object-oriented programming languages consists of 48
> (47 + an "unknown type") different change types … describe a meaningful action performed by a
> developer in a commit (e.g., statement_delete, statement_insert, removed_class,
> additional_class)*

Three things follow, and they shape everything after.

**They worked per file, then summed.** The tool ran on file pairs; the commit's feature vector is
the total across its files. Their aggregation unit is the file.

**Their features have no length limit.** A commit with 598 `STATEMENT_INSERT`s is the number 598.
A commit touching 300 files and one touching 2 both produce complete 48-value vectors. Nothing is
ever cut off.

**Direction and element type are the features.** `statement_insert` and `statement_delete` are
different types; so are `method_renaming` and `attribute_renaming`. Whether a line was added or
removed, and what kind of element it belonged to, is not formatting around the signal — it *is*
the signal.

That last point answers question 4 before we get there, and the first two say what an equivalent
architecture has to do: cover the whole commit, and aggregate over files.

In [3]:
# What the dataset's change-type columns look like, as a reminder of the target.
_cols = require("levin_columns.json", optional=True)
cols = json.load(open(_cols)) if _cols else None
if cols:
    ch = cols["changes"]
    print(f"{len(ch)} change-type features per commit, e.g.\n")
    for c in sorted(ch)[:8]:
        print(f"   {c}")
    print(f"   ... and {len(ch)-8} more")
    print("\nA fixed-length summary of an unbounded commit. That is what CodeBERT has to match.")

## 3. How do we mimic that with CodeBERT?

Levin's shape was already split-encode-aggregate: run per file, sum across files. The equivalent
is to encode each file's change separately and aggregate the embeddings — so the measurement that
matters is **tokens per file**, not per commit.

In [4]:
def structured_pieces(rec):
    """One piece for the message, one per changed file. DRS-OSS style markers."""
    out = [f"<COMMIT_MESSAGE>{message_of(rec)}</COMMIT_MESSAGE>"]
    for f in rec.get("files") or []:
        added, removed = [], []
        for ln in (f.get("patch") or "").split("\n"):
            if HUNK.match(ln) or ln.startswith(("--- ", "+++ ")):
                continue
            if ln.startswith("+"):
                added.append(ln[1:])
            elif ln.startswith("-"):
                removed.append(ln[1:])
        piece = [f"<FILE>{f.get('filename')}</FILE>"]
        if removed:
            piece.append("<REMOVED>" + "\n".join(removed) + "</REMOVED>")
        if added:
            piece.append("<ADDED>" + "\n".join(added) + "</ADDED>")
        out.append("\n".join(piece))
    return out


pieces = R.apply(structured_pieces, axis=1)
per_piece = pd.Series([ntok(p) for ps in pieces for p in ps])
n_pieces = pieces.map(len)

print("TOKENS PER PIECE\n")
print(per_piece.describe(percentiles=[.5, .75, .9, .95, .99]).round(0).to_string())
print(f"\n   pieces within {LIMIT} tokens : {(per_piece <= LIMIT).mean()*100:.1f}%")
print(f"\npieces per commit: median {int(n_pieces.median())}, "
      f"90th {int(n_pieces.quantile(.9))}, max {int(n_pieces.max())}")
print(f"""
So a piece usually fits where a commit usually does not. Pieces that are themselves too long are
split and pooled within the piece, so nothing is discarded at any level.""")

TOKENS PER PIECE

count     7181.0
mean       521.0
std       1405.0
min         16.0
50%        161.0
75%        454.0
90%       1127.0
95%       1966.0
99%       5283.0
max      36095.0

   pieces within 512 tokens : 77.5%

pieces per commit: median 3, 90th 13, max 248

So a piece usually fits where a commit usually does not. Pieces that are themselves too long are
split and pooled within the piece, so nothing is discarded at any level.


### What aggregation costs, and what it risks

Encoding is cheap. The risk is in the combining step: pooling many embeddings into one vector
loses more the more there are to pool.

In [5]:
passes = int(np.ceil(per_piece / LIMIT).clip(lower=1).sum())
naive = int(np.ceil(raw_tokens / LIMIT).sum())
print(f"forward passes, per-file : {passes:,}")
print(f"forward passes, chunking : {naive:,}")
print(f"commits                  : {len(R):,}")
print("""
Per-file costs more, not less -- most files are small, so each pass is underfilled. The reason to
prefer it is that the unit matches how the labels were produced, not efficiency. Both are minutes
on a GPU for a 125M model.

THE RISK IN AGGREGATION""")
for q in (0.5, 0.9, 0.99, 1.0):
    print(f"   {int(q*100):>3}th percentile: {int(n_pieces.quantile(q)):>4} pieces to pool")
print("""
Pooling three embeddings preserves detail; pooling two hundred averages it away. The distribution
is kind at the median and brutal in the tail, which makes the choice of pooling function a real
decision rather than a default -- Stage 4 compares mean, max and attention-weighted.""")

forward passes, per-file : 12,071
forward passes, chunking : 11,496
commits                  : 1,150

Per-file costs more, not less -- most files are small, so each pass is underfilled. The reason to
prefer it is that the unit matches how the labels were produced, not efficiency. Both are minutes
on a GPU for a 125M model.

THE RISK IN AGGREGATION
    50th percentile:    3 pieces to pool
    90th percentile:   13 pieces to pool
    99th percentile:   43 pieces to pool
   100th percentile:  248 pieces to pool

Pooling three embeddings preserves detail; pooling two hundred averages it away. The distribution
is kind at the median and brutal in the tail, which makes the choice of pooling function a real
decision rather than a default -- Stage 4 compares mean, max and attention-weighted.


## 4. Why not just keep the raw code changes and drop everything else?

That is a real option — it is the `no-context` representation below: added and removed lines only,
with hunk headers, context lines, prefixes and file paths all removed.

Two reasons it is not the answer.

In [6]:
def rep_nocontext(rec):
    parts = [message_of(rec)]
    for f in rec.get("files") or []:
        for ln in (f.get("patch") or "").split("\n"):
            if not HUNK.match(ln) and not ln.startswith(("--- ", "+++ ")) \
                    and ln.startswith(("+", "-")):
                parts.append(ln[1:])
    return "\n".join(parts)


def rep_structured(rec):
    return "\n".join(structured_pieces(rec))


T = pd.DataFrame({
    "raw": raw_tokens,
    "no-context (changes only)": R.apply(lambda r: ntok(rep_nocontext(r)), axis=1),
    "structured": R.apply(lambda r: ntok(rep_structured(r)), axis=1),
})

print("REASON ONE -- it does not solve the problem\n")
print(pd.DataFrame({
    "median tokens": T.median().round(0),
    f"over {LIMIT}": (T > LIMIT).mean().mul(100).round(1),
}).to_string())

share = (T["no-context (changes only)"] / T["raw"]).median()
print(f"""
The changed lines alone are {share*100:.0f}% of a raw diff in the median commit. Stripping
everything else saves {(1-share)*100:.0f}%, and still leaves most commits over the limit. The
commits are simply large -- this is not an encoding problem that a leaner format fixes.""")

print("\n\nREASON TWO -- it discards what the taxonomy is built on\n")
ex = R.iloc[int(np.argmin(np.abs(T["structured"] - T["structured"].median())))]
print("structured, keeps direction and file:\n")
print("   " + rep_structured(ex)[:300].replace("\n", "\n   "))
print("\n\nchanges only, direction and file gone:\n")
print("   " + rep_nocontext(ex)[:300].replace("\n", "\n   "))
print("""
Levin's taxonomy separates statement_insert from statement_delete, and method_renaming from
attribute_renaming. Strip the +/- and a model cannot tell an insertion from a deletion; strip the
file path and it cannot tell which element was touched. Those distinctions are not decoration
around the change -- they are the 48 categories being replicated.

Structured costs a few points of token budget over changes-only and keeps both.""")

REASON ONE -- it does not solve the problem

                           median tokens  over 512
raw                               1670.0      79.8
no-context (changes only)          823.0      60.7
structured                         998.0      67.0

The changed lines alone are 50% of a raw diff in the median commit. Stripping
everything else saves 50%, and still leaves most commits over the limit. The
commits are simply large -- this is not an encoding problem that a leaner format fixes.


REASON TWO -- it discards what the taxonomy is built on

structured, keeps direction and file:

   <COMMIT_MESSAGE>YARN-2111. In FairScheduler.attemptScheduling, we don't count containers as assigned if they have 0 memory but non-zero cores (Sandy Ryza)
   
   git-svn-id: https://svn.apache.org/repos/asf/hadoop/common/branches/branch-2@1605115 13f79535-47bb-0310-9956-ffa450edef68</COMMIT_MESSAGE>
   <FILE


changes only, direction and file gone:

   YARN-2111. In FairScheduler.attemptScheduling, we d

## 5. Decision

In [7]:
piece_fit = (per_piece <= LIMIT).mean() * 100
commit_fit = (T["structured"] <= LIMIT).mean() * 100

print("=" * 74)
print("STAGE 3.5 DECISION")
print("=" * 74)
print(f"""
1. IS 512 ENOUGH?
   No. {100-commit_fit:.1f}% of commits exceed it as a single input, and the largest is
   {int(raw_tokens.max()):,} tokens.

2. HOW DID LEVIN USE CHANGEDISTILLER?
   Per file pair, summed across the commit, producing 48 counts with no length limit.

3. HOW DO WE MIMIC IT?
   Encode each file's change separately, aggregate across files. {piece_fit:.1f}% of pieces fit in
   one pass; longer pieces are split and pooled within the piece. Nothing is truncated.

4. WHY NOT JUST THE CODE CHANGES?
   Because it does not fit either ({(T['no-context (changes only)'] > LIMIT).mean()*100:.1f}% still
   over), and because dropping the +/- and the file path removes exactly what separates
   statement_insert from statement_delete.

REPRESENTATION   structured (DRS-OSS markers)
ARCHITECTURE     per-file encoding, aggregated across pieces
MODEL            CodeBERT, 125M parameters, code-pretrained

   The binding constraint is 1,151 training examples, not context length. A larger long-context
   model would take whole commits in one pass, but fine-tuning it on this many examples overfits.
   Coverage by context length is below, so the option can be reconsidered on evidence if Stage 4's
   CodeBERT results disappoint.

STAGE 4 MUST DECIDE   pooling function: mean, max, or attention-weighted. Median 3 pieces to pool,
   but up to {int(n_pieces.max())} -- mean is the default and likely the worst of the three.
""")

print("COVERAGE IF A SINGLE PASS WERE USED INSTEAD\n")
for L in (512, 2048, 8192, 32768, 131072):
    print(f"   {L:>7,} tokens : {(T['structured'] <= L).mean()*100:5.1f}% of commits")

json.dump({
    "representation": "structured",
    "architecture": "per-file encoding with aggregation",
    "model": MODEL,
    "limit": LIMIT,
    "pct_commits_within_limit": round(float(commit_fit), 2),
    "pct_pieces_within_limit": round(float(piece_fit), 2),
    "median_pieces_per_commit": int(n_pieces.median()),
    "max_pieces_per_commit": int(n_pieces.max()),
    "forward_passes": passes,
    "coverage_by_context": {str(L): round(float((T["structured"] <= L).mean()*100), 2)
                            for L in (512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072)},
    "stage4_open": ["pooling function: mean | max | attention-weighted"],
}, open("stage3b_decision.json", "w"), indent=1)

pd.DataFrame({"commit_id": R["commit_id"], "label": R["label"],
              "n_pieces": n_pieces, **{k: v for k, v in T.items()}}) \
  .to_csv("stage3b_token_counts.csv", index=False)
print("\n   stage3b_decision.json and stage3b_token_counts.csv written")

STAGE 3.5 DECISION

1. IS 512 ENOUGH?
   No. 67.0% of commits exceed it as a single input, and the largest is
   90,669 tokens.

2. HOW DID LEVIN USE CHANGEDISTILLER?
   Per file pair, summed across the commit, producing 48 counts with no length limit.

3. HOW DO WE MIMIC IT?
   Encode each file's change separately, aggregate across files. 77.5% of pieces fit in
   one pass; longer pieces are split and pooled within the piece. Nothing is truncated.

4. WHY NOT JUST THE CODE CHANGES?
   Because it does not fit either (60.7% still
   over), and because dropping the +/- and the file path removes exactly what separates
   statement_insert from statement_delete.

REPRESENTATION   structured (DRS-OSS markers)
ARCHITECTURE     per-file encoding, aggregated across pieces
MODEL            CodeBERT, 125M parameters, code-pretrained

   The binding constraint is 1,151 training examples, not context length. A larger long-context
   model would take whole commits in one pass, but fine-tuning it on 